# Exercício 5.5 — Sinônimos, Paráfrases e Embeddings

Exercício do *Mini-Lab: Embeddings e Vetores com BERT* (seção 5.5).

**Objetivo:** comparar o buscador **TF‑IDF** vs **BERT** (CLS e MEAN pooling) quando a consulta usa **sinônimos** ou **paráfrases** — ou seja, quando as palavras exatas não batem.

## Contexto

No mini‑lab, o TF‑IDF depende muito de **palavras iguais**: se a consulta usa sinônimos/paráfrases, ele pode falhar. Já embeddings densos (BERT/SBERT) tentam capturar **significado**, não apenas termos.

Este notebook é autocontido: ele carrega o mesmo dataset do mini‑lab e reproduz as funções de busca. Execute as células em ordem.

## 1) Imports e utilitários

In [1]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import torch
from transformers import AutoTokenizer, AutoModel

\\wsl$\Ubuntu\home\guilherme\projetos\disciplinas\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def l2_norm(v: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    denom = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / np.maximum(denom, eps)

def top_k_similar(query_vec: np.ndarray, matrix_vecs: np.ndarray, k: int = 5):
    sims = cosine_similarity(query_vec.reshape(1, -1), matrix_vecs).flatten()
    idx = np.argsort(-sims)[:k]
    return idx, sims[idx]

def show_results(query: str, texts: list[str], idx: np.ndarray, scores: np.ndarray, title: str):
    print('=' * 90)
    print(title)
    print(f'QUERY: {query}\n')
    for rank, (i, s) in enumerate(zip(idx, scores), start=1):
        print(f'[{rank}] score={s:.4f}  |  id={i}')
        print(f'    {texts[i]}')
        print('-' * 90)

## 2) Dataset (mesmo do mini‑lab)

In [ ]:
texts = [
    'O Kubernetes organiza aplicações em contêineres usando pods e namespaces.',
    'BERT é um modelo de linguagem baseado em Transformers com atenção bidirecional.',
    'TF-IDF representa textos como vetores esparsos de termos com pesos.',
    'O Keycloak é uma solução de identidade para autenticação e single sign-on (SSO).',
    'Uma rede neural pode aprender representações úteis para classificação de texto.',
    'Esteatose hepática é o acúmulo de gordura no fígado e pode ser associada a obesidade.',
    'Exercícios físicos regulares ajudam a melhorar a saúde cardiovascular.',
    'Uma dieta balanceada inclui proteínas, carboidratos, fibras e gorduras saudáveis.',
    'Em Paris, o metrô e o RER conectam pontos turísticos e regiões metropolitanas.',
    'Em Budapeste, os banhos termais são uma atração tradicional muito conhecida.',
    'A culinária francesa inclui pratos como confit de pato e magret de pato.',
    'O t-SNE projeta dados de alta dimensão para 2D preservando vizinhanças locais.',
    'Embeddings transformam texto em vetores densos que capturam semântica.',
    'Similaridade cosseno mede o alinhamento entre dois vetores.',
    'O SharePoint Online permite criar sites, listas e páginas para intranet corporativa.',
    'Power Automate automatiza fluxos de trabalho integrando serviços do Microsoft 365.',
    'Uma VPN cria um túnel criptografado para acessar recursos internos remotamente.',
    'RAG combina recuperação de documentos com geração de texto por LLMs.',
    'O MinIO é um storage compatível com S3 para objetos.',
    'Milvus é um banco de dados vetorial para busca por similaridade.',
    'O GitHub Actions executa pipelines de CI/CD para build e deploy.',
    'A normalização L2 ajusta vetores para terem norma 1.',
    'O produto escalar soma multiplicações elemento a elemento entre dois vetores.',
    'Uma eSIM permite usar planos móveis sem um chip físico tradicional.',
    'Um fone com cancelamento de ruído reduz sons externos usando processamento de sinal.',
    'Um condomínio possui área privativa e área de uso comum em sua composição.',
    'O risco de duplicação de eventos pode ser evitado guardando IDs em uma planilha.',
    'O termo underfitting indica um modelo simples demais para capturar padrões.',
    'A técnica de mean pooling tira a média dos embeddings dos tokens de uma frase.',
    'O token [CLS] pode ser usado como representação agregada em modelos BERT.',
]

df = pd.DataFrame({'id': range(len(texts)), 'text': texts})
df.head()

## 3) Buscador TF‑IDF (baseline)

In [ ]:
tfidf = TfidfVectorizer(lowercase=True)
X_tfidf = tfidf.fit_transform(texts)
X_tfidf.shape

In [ ]:
def search_tfidf(query: str, k: int = 5):
    q_vec = tfidf.transform([query])
    sims = cosine_similarity(q_vec, X_tfidf).flatten()
    idx = np.argsort(-sims)[:k]
    return idx, sims[idx]

## 4) Buscador BERT (CLS e MEAN pooling)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
bert_model_name = 'bert-base-multilingual-cased'
tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert = AutoModel.from_pretrained(bert_model_name).to(device)
bert.eval()
print('Loaded:', bert_model_name)

In [ ]:
@torch.no_grad()
def bert_encode(text_list: list[str], pooling: str = 'mean', max_length: int = 64, batch_size: int = 16) -> np.ndarray:
    all_vecs = []
    for start in range(0, len(text_list), batch_size):
        batch = text_list[start:start + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt').to(device)
        out = bert(**enc)
        last_hidden = out.last_hidden_state
        if pooling == 'cls':
            sent_vec = last_hidden[:, 0, :]
        elif pooling == 'mean':
            mask = enc['attention_mask'].unsqueeze(-1).type_as(last_hidden)
            summed = (last_hidden * mask).sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1e-9)
            sent_vec = summed / counts
        else:
            raise ValueError("pooling must be 'cls' or 'mean'")
        all_vecs.append(sent_vec.cpu().numpy())
    return np.vstack(all_vecs)

In [ ]:
X_bert_cls = bert_encode(texts, pooling='cls')
X_bert_mean = bert_encode(texts, pooling='mean')
X_bert_cls_n = l2_norm(X_bert_cls)
X_bert_mean_n = l2_norm(X_bert_mean)
X_bert_cls.shape, X_bert_mean.shape

In [ ]:
def search_bert(query: str, X: np.ndarray, pooling_name: str, k: int = 5):
    q_vec = bert_encode([query], pooling='mean')[0]
    q_vec = l2_norm(q_vec.reshape(1, -1))[0]
    idx, scores = top_k_similar(q_vec, X, k=k)
    show_results(query, texts, idx, scores, title=f'BERT Search ({pooling_name})')

## 5) O exercício (5.5)

Teste consultas com **sinônimos** e **paráfrases** e compare as três abordagens:

- `"banco vetorial"`
- `"vector database"` (inglês!)
- `"busca por similaridade"`

Execute as células abaixo e observe o ranking de cada método.

In [ ]:
query_exemplos = [
    'banco vetorial',
    'vector database',
    'busca por similaridade',
    'como comparar vetores?',
    'o que é SSO e autenticação?',
]

for q in query_exemplos:
    print('#' * 100)
    print(f'QUERY: {q}')
    print('#' * 100 + '\n')
    idx, scores = search_tfidf(q, k=3)
    show_results(q, texts, idx, scores, title='TF-IDF')
    search_bert(q, X_bert_cls_n, 'CLS', k=3)
    search_bert(q, X_bert_mean_n, 'MEAN', k=3)

## 6) Responda

1) Qual abordagem retornou itens mais **"semânticos"** para as consultas com sinônimos?

R.:


2) Qual abordagem pareceu **presa às palavras exatas**?

R.:


3) Para a consulta `"vector database"` (em inglês), o que acontece em cada método? O BERT multilíngue consegue relacionar com textos em português?

R.:

## 7) Teste suas próprias consultas

Crie mais consultas com sinônimos/paráfrases (ex.: "armazenamento de objetos", "logins centralizados", "perder peso") e rode o comparativo abaixo.

In [ ]:
minhas_queries = [
    'armazenamento de objetos',
    # adicione mais aqui
]

for q in minhas_queries:
    print('#' * 100)
    print(f'QUERY: {q}')
    print('#' * 100 + '\n')
    idx, scores = search_tfidf(q, k=3)
    show_results(q, texts, idx, scores, title='TF-IDF')
    search_bert(q, X_bert_cls_n, 'CLS', k=3)
    search_bert(q, X_bert_mean_n, 'MEAN', k=3)